In [ ]:
import os
import re
import shap
import time
import pickle
import textstat
import numpy as np
import pandas as pd
import missingno as msno
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from plotly.colors import n_colors
from collections import namedtuple
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from xgboost import XGBRegressor, XGBClassifier
from scipy.stats import kruskal, shapiro, spearmanr, kendalltau
from sklearn.model_selection import RandomizedSearchCV, LearningCurveDisplay, learning_curve, GridSearchCV

In [ ]:
random_state = 42

tool_wo_accepted_answer = {
    'Domino', 
    'DVC', 
    'Guild AI"', 
    'MLflow', 
    'Polyaxon', 
    'SigOpt'
}

challenge_type_mapping = {
    'problem': 0, 
    'knowledge': 1
}


In [ ]:
path_result = '../../Result'

path_rq2 = os.path.join(path_result, 'RQ3')
path_rq3 = os.path.join(path_result, 'RQ3')


In [ ]:
# name = 'labels'
# path = path_rq3
# df = pd.read_json(os.path.join(f'{path}', f'{name}.json'))

# for index, row in df.iterrows():
#     if 'Git' in row['Platform']:
#         df.at[index, 'Solution_body'] = row['Discussion_body']
#         df.at[index, 'Solution_score_count'] = row['Discussion_score_count']
        
# del df['Discussion_body']
# del df['Discussion_score_count']

# df.to_json(os.path.join(f'{path}', f'{name}.json'), indent=4, orient='records')

In [ ]:
# df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
# try:
#     del df['Answerer_location']
# except:
#     pass
# try:
#     del df['Answerer_created_time']
# except:
#     pass
# try:
#     del df['Answerer_last_access_time']
# except:
#     pass
# try:
#     del df['Poster_created_time']
# except:
#     pass
# try:
#     del df['Poster_last_access_time']
# except:
#     pass
# try:
#     del df['Poster_location']
# except:
#     pass
# df.to_json(os.path.join(path_rq3, 'metrics.json'), indent=4, orient='records')

In [ ]:
df = pd.read_json(os.path.join(path_rq2, 'mapped.json'))

for index, row in df.iterrows():
    creation_time = row['Challenge_created_time']
    closed_time = row['Challenge_closed_time']
    if pd.notna(creation_time) and pd.notna(closed_time) and (closed_time > creation_time):
        df.at[index, 'Challenge_solved_time'] = (closed_time - creation_time) / pd.Timedelta(hours=1)
    
    df.at[index, 'Challenge_edited'] = True if pd.notna(row['Challenge_last_edit_time']) else False
    df.at[index, 'Solution_edited'] = True if pd.notna(row['Solution_last_edit_time']) else False

df['Challenge_contributor_issue_ratio'] = df['Challenge_repo_contributor_count'] / df['Challenge_repo_issue_count']
df['Challenge_watch_issue_ratio'] = df['Challenge_repo_watch_count'] / df['Challenge_repo_issue_count']
# df['Challenge_type'] = df['Challenge_type'].map(challenge_type_mapping)

# df = df[df.columns.drop(list(df.filter(regex=r'title|body|created_time|closed_time|edit_time|topic|summary|link|root|')))]
df = df.reindex(sorted(df.columns), axis=1)
df.to_json(os.path.join(path_rq3, 'metrics.json'), indent=4, orient='records')


In [ ]:
df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Platform'].str.contains('Git')]
print(df['Tool'].value_counts())

# df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
# df = df[df['Platform'].str.contains('Stack Overflow')]
# print(df['Tool'].value_counts())

# df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
# df = df[df['Platform'].str.contains('Tool-specific')]
# df = df[~df['Tool'].isin(tool_wo_accepted_answer)]
# print(df['Tool'].value_counts())

In [ ]:
categorical_columns = ['Challenge_type', 'Challenge_topic_macro', 'Challenge_summary_topic_macro', 'Challenge_edited']
numerical_columns = ['Challenge_link_count_documentation', 'Challenge_link_count_example', 'Challenge_link_count_image', 'Challenge_link_count_issue', 'Challenge_link_count_patch', 'Challenge_link_count_tool', 'Challenge_link_count_tutorial', 'Challenge_readability', 'Challenge_reading_time', 'Challenge_word_count', 'Challenge_score_count']
numerical_columns_git = ['Challenge_comment_count', 'Challenge_contributor_issue_ratio', 'Challenge_repo_contributor_count', 'Challenge_repo_fork_count', 'Challenge_repo_issue_count', 'Challenge_repo_star_count', 'Challenge_repo_watch_count', 'Challenge_watch_issue_ratio']
numerical_columns_git = numerical_columns + numerical_columns_git
numerical_columns_so = ['Challenge_answer_count', 'Challenge_comment_count', 'Challenge_favorite_count', 'Challenge_view_count', 'Poster_reputation_count', 'Poster_view_count']
numerical_columns_so = numerical_columns + numerical_columns_so
numerical_columns_to_basic = ['Challenge_answer_count']
numerical_columns_to_customized = {
    'Amazon SageMaker': ['Challenge_comment_count', 'Challenge_favorite_count', 'Poster_isAwsEmployee', 'Poster_isCse', 'Poster_isExpert', 'Poster_isModerator', 'Challenge_view_count'],
    'Azure Machine Learning': ['Challenge_comment_count'],
    'Guild AI': ['Challenge_view_count'],
    'Vertex AI': ['Challenge_view_count'],
    'Weights & Biases': ['Challenge_view_count'],
}
numerical_columns_to = set(val for sublist in numerical_columns_to_customized.values() for val in sublist)
numerical_columns_to = list(numerical_columns_to.union(numerical_columns_to_basic))


In [ ]:
param_grid = {
    'n_estimators': [100, 500, 1000],
    'max_depth': [3, 6, 9],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'objective': ['binary:logistic'],
    'eval_metric': ['auc'],
    'random_state': [random_state],
    'tree_method': ['gpu_hist'],
    'gpu_id': [1],
}

df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=np.nan
)
numerical_scaler = RobustScaler()
preprocessor = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_scaler, numerical_columns),
    ],
    verbose_feature_names_out=False,
)

X = df[categorical_columns + numerical_columns]
X_preprocessed = preprocessor.fit_transform(X)
y = df['Challenge_solved_time'].isna()

classifier = XGBClassifier()
grid_search = RandomizedSearchCV(classifier, param_grid, n_jobs=-1, n_iter=1)
grid_search.fit(X_preprocessed,y)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 10), sharey=True)
train_sizes, train_scores, test_scores = learning_curve(grid_search.best_estimator_, X_preprocessed, y)
display = LearningCurveDisplay(train_sizes=train_sizes, train_scores=train_scores, test_scores=test_scores, score_name="Score")
display.plot()

In [ ]:
df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Platform'].str.contains('Git')]

param_grid = {
    'n_estimators': [20, 40, 60, 80, 100],
    'max_depth': [3, 6, 9],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'objective': ['binary:logistic'],
    'eval_metric': ['auc'],
    'random_state': [random_state],
    'tree_method': ['gpu_hist'],
    'gpu_id': [1],
}

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=np.nan
)
numerical_scaler = RobustScaler()
preprocessor = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_scaler, numerical_columns_git),
    ],
    verbose_feature_names_out=False,
)
    
X = df[categorical_columns + numerical_columns_git]
X_preprocessed = preprocessor.fit_transform(X)
y = df['Challenge_solved_time'].isna()

classifier = XGBClassifier()
random_search = RandomizedSearchCV(classifier, param_grid, n_jobs=-1, random_state=random_state, n_iter=5)
random_search.fit(X_preprocessed,y)
    
name = 'Overall'
print(f'{name} ROC-AUC: {random_search.best_score_}')

explainer = shap.TreeExplainer(random_search.best_estimator_)
# shap_values = explainer(X_preprocessed).values.mean(0)
# shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
# feature_importance = pd.Series(
#     shap_values,
#     index = X.columns,
# ).to_frame(name)

shap_values = explainer(X_preprocessed)
shap.summary_plot(shap_values, X)#, show=False)
# plt.savefig(os.path.join(path_difficulty,
#             f'Challenge_openness shap_based_feature_importance (Git).png'), bbox_inches='tight')

# roc_auc = []

# for name, group in df.groupby('Tool'):
#     if len(group) < 50:
#         continue

#     categorical_encoder = OrdinalEncoder(
#         handle_unknown="use_encoded_value", unknown_value=np.nan
#     )
#     numerical_scaler = RobustScaler()
#     preprocessor = ColumnTransformer(
#         [
#             ("cat", categorical_encoder, categorical_columns),
#             ("num", numerical_scaler, numerical_columns_git),
#         ],
#         verbose_feature_names_out=False,
#     )
    
#     X = group[categorical_columns + numerical_columns_git]
#     X_preprocessed = preprocessor.fit_transform(X)
#     y = group['Challenge_solved_time'].isna()

#     classifier = XGBClassifier()
#     random_search = RandomizedSearchCV(classifier, param_grid, n_jobs=-1, random_state=random_state)
#     random_search.fit(X_preprocessed,y)
    
#     roc_auc.append(random_search.best_score_)
#     print(f'{name}: {random_search.best_score_}')
    
#     explainer = shap.TreeExplainer(random_search.best_estimator_)
#     shap_values = explainer(X_preprocessed).values.mean(0)
#     shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
#     importances = pd.Series(
#         shap_values,
#         index = X.columns,
#     ).to_frame(name)
#     feature_importance = pd.concat([feature_importance, importances], axis=1)

# print(f"Average ROC-AUC: {sum(roc_auc)/len(roc_auc)}")
# fig = px.imshow(feature_importance, title='Challenge openness feature importance (Git)', color_continuous_scale='RdBu', width=1000, height=1000)
# fig.write_image(os.path.join(path_rq3, 'Challenge openness feature importance (Git).png'))


In [ ]:
df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Platform'].str.contains('Stack Overflow')]

param_grid = {
    'n_estimators': [100, 500, 1000],
    'max_depth': [6, 9, 12, 15],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'objective': ['binary:logistic'],
    'eval_metric': ['auc'],
    'random_state': [random_state],
    'tree_method': ['gpu_hist'],
    'gpu_id': [1],
}

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=np.nan
)
numerical_scaler = RobustScaler()
preprocessor = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_scaler, numerical_columns_so),
    ],
    verbose_feature_names_out=False,
)
    
X = df[categorical_columns + numerical_columns_so]
X_preprocessed = preprocessor.fit_transform(X)
y = df['Challenge_solved_time'].isna()

classifier = XGBClassifier()
random_search = RandomizedSearchCV(classifier, param_grid, n_jobs=-1, random_state=random_state)
random_search.fit(X_preprocessed,y)
    
name = 'Overall'
print(f'{name} ROC-AUC: {random_search.best_score_}')
    
explainer = shap.TreeExplainer(random_search.best_estimator_)
shap_values = explainer(X_preprocessed).values.mean(0)
shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
feature_importance = pd.Series(
    shap_values,
    index = X.columns,
).to_frame(name)

roc_auc = []

for name, group in df.groupby('Tool'):
    if len(group) < 50:
        continue

    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=np.nan
    )
    numerical_scaler = RobustScaler()
    preprocessor = ColumnTransformer(
        [
            ("cat", categorical_encoder, categorical_columns),
            ("num", numerical_scaler, numerical_columns_so),
        ],
        verbose_feature_names_out=False,
    )
    
    X = group[categorical_columns + numerical_columns_so]
    X_preprocessed = preprocessor.fit_transform(X)
    y = group['Challenge_solved_time'].isna()

    classifier = XGBClassifier()
    random_search = RandomizedSearchCV(classifier, param_grid, n_jobs=-1, random_state=random_state)
    random_search.fit(X_preprocessed,y)
    
    roc_auc.append(random_search.best_score_)
    print(f'{name}: {random_search.best_score_}')
    
    explainer = shap.TreeExplainer(random_search.best_estimator_)
    shap_values = explainer(X_preprocessed).values.mean(0)
    shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
    importances = pd.Series(
        shap_values,
        index = X.columns,
    ).to_frame(name)
    feature_importance = pd.concat([feature_importance, importances], axis=1)

print(f"Average ROC-AUC: {sum(roc_auc)/len(roc_auc)}")
fig = px.imshow(feature_importance, title='Challenge openness feature importance (Stack Overflow)', color_continuous_scale='RdBu', width=1000, height=1000)
fig.write_image(os.path.join(path_rq3, 'Challenge openness feature importance (Stack Overflow).png'))


In [ ]:
df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Platform'].str.contains('Tool-specific')]
df = df[~df['Tool'].isin(tool_wo_accepted_answer)]

param_grid = {
    'n_estimators': [100, 500, 1000],
    'max_depth': [6, 9, 12, 15],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'objective': ['binary:logistic'],
    'eval_metric': ['auc'],
    'random_state': [random_state],
    'tree_method': ['gpu_hist'],
    'gpu_id': [1],
}

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=np.nan
)
numerical_scaler = RobustScaler()
preprocessor = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_scaler, numerical_columns_to),
    ],
    verbose_feature_names_out=False,
)
    
X = df[categorical_columns + numerical_columns_to]
X_preprocessed = preprocessor.fit_transform(X)
y = df['Challenge_solved_time'].isna()

classifier = XGBClassifier()
random_search = RandomizedSearchCV(classifier, param_grid, n_jobs=-1, random_state=random_state)
random_search.fit(X_preprocessed,y)
    
name = 'Overall'
print(f'{name} ROC-AUC: {random_search.best_score_}')
    
explainer = shap.TreeExplainer(random_search.best_estimator_)
shap_values = explainer(X_preprocessed).values.mean(0)
shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
feature_importance = pd.Series(
    shap_values,
    index = X.columns,
).to_frame(name)

roc_auc = []

for name, group in df.groupby('Tool'):
    if len(group) < 50:
        continue

    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=np.nan
    )
    numerical_scaler = RobustScaler()
    numerical_columns = numerical_columns_to_basic
    if name in numerical_columns_to_customized:
        numerical_columns = numerical_columns + numerical_columns_to_customized[name]
    preprocessor = ColumnTransformer(
        [
            ("cat", categorical_encoder, categorical_columns),
            ("num", numerical_scaler, numerical_columns),
        ],
        verbose_feature_names_out=False,
    )
    
    X = group[categorical_columns + numerical_columns]
    X_preprocessed = preprocessor.fit_transform(X)
    y = group['Challenge_solved_time'].isna()

    classifier = XGBClassifier()
    random_search = RandomizedSearchCV(classifier, param_grid, n_jobs=-1, random_state=random_state)
    random_search.fit(X_preprocessed,y)
    
    roc_auc.append(random_search.best_score_)
    print(f'{name}: {random_search.best_score_}')
    
    explainer = shap.TreeExplainer(random_search.best_estimator_)
    shap_values = explainer(X_preprocessed).values.mean(0)
    shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
    importances = pd.Series(
        shap_values,
        index = X.columns,
    ).to_frame(name)
    feature_importance = pd.concat([feature_importance, importances], axis=1)

print(f"Average ROC-AUC: {sum(roc_auc)/len(roc_auc)}")
feature_importance = feature_importance.fillna(0)
fig = px.imshow(feature_importance, title='Challenge openness feature importance (Tool-specific)', color_continuous_scale='RdBu', width=1000, height=1000)
fig.write_image(os.path.join(path_rq3, 'Challenge openness feature importance (Tool-specific).png'))


In [ ]:
# df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
# df = df[df['Challenge_solved_time'].notna()]
# df = df[df['Platform'].str.contains('Git')]
# print(df['Tool'].value_counts())

# df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
# df = df[df['Challenge_solved_time'].notna()]
# df = df[df['Platform'].str.contains('Stack Overflow')]
# print(df['Tool'].value_counts())

# df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
# df = df[df['Challenge_solved_time'].notna()]
# df = df[df['Platform'].str.contains('Tool-specific')]
# df = df[~df['Tool'].isin(tool_wo_accepted_answer)]
# print(df['Tool'].value_counts())

In [ ]:
categorical_columns = ['Challenge_type', 'Challenge_topic_macro', 'Challenge_summary_topic_macro', 'Solution_summary_topic_macro', 'Challenge_root_cause_summary_topic_macro', 'Challenge_edited', 'Solution_edited', 'Challenge_self_closed']
numerical_columns = ['Challenge_link_count', 'Challenge_readability', 'Challenge_reading_time', 'Challenge_word_count', 'Challenge_score_count', 'Solution_link_count', 'Solution_link_docs', 'Solution_link_examples', 'Solution_link_issues', 'Solution_link_patches', 'Solution_link_tools', 'Solution_link_tutorials', 'Solution_readability', 'Solution_reading_time', 'Solution_word_count']
numerical_columns_git = ['Challenge_comment_count', 'Challenge_contributor_issue_ratio', 'Challenge_repo_contributor_count', 'Challenge_repo_fork_count', 'Challenge_repo_issue_count', 'Challenge_repo_star_count', 'Challenge_repo_watch_count', 'Challenge_watch_issue_ratio', 'Solution_score_count']
numerical_columns_git = numerical_columns + numerical_columns_git
numerical_columns_so = ['Challenge_answer_count', 'Challenge_comment_count', 'Challenge_favorite_count', 'Challenge_view_count', 'Poster_reputation_count', 'Poster_view_count', 'Solution_comment_count', 'Answerer_reputation_count', 'Answerer_view_count', 'Solution_score_count']
numerical_columns_so = numerical_columns + numerical_columns_so
numerical_columns_to_basic = ['Challenge_answer_count']
numerical_columns_to_customized = {
    'Amazon SageMaker': ['Challenge_comment_count', 'Challenge_favorite_count', 'Poster_isAwsEmployee', 'Poster_isCse', 'Poster_isExpert', 'Poster_isModerator', 'Challenge_view_count', 'Solution_score_count', 'Answerer_isAwsEmployee', 'Answerer_isCse', 'Answerer_isExpert', 'Answerer_isModerator'],
    'Azure Machine Learning': ['Challenge_comment_count', 'Solution_score_count'],
    'Guild AI': ['Challenge_view_count'],
    'Vertex AI': ['Challenge_view_count'],
    'Weights & Biases': ['Challenge_view_count'],
}
numerical_columns_to = set(val for sublist in numerical_columns_to_customized.values() for val in sublist)
numerical_columns_to = list(numerical_columns_to.union(numerical_columns_to_basic))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import LearningCurveDisplay, learning_curve

param_grid = {
    'n_estimators': [100, 500, 1000],
    'max_depth': [3, 6, 9],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'objective': ['reg:squarederror'],
    'random_state': [random_state],
    'tree_method': ['gpu_hist'],
    'gpu_id': [3],
}

df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Challenge_solved_time'].notna()]

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=np.nan
)
numerical_scaler = RobustScaler()
preprocessor = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_scaler, numerical_columns),
    ],
    verbose_feature_names_out=False,
)

X = df[categorical_columns + numerical_columns]
X_preprocessed = preprocessor.fit_transform(X)
y = df['Challenge_solved_time']

regressor = XGBRegressor()
grid_search = GridSearchCV(regressor, param_grid, n_jobs=-1)
grid_search.fit(X_preprocessed,y)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 10), sharey=True)
train_sizes, train_scores, test_scores = learning_curve(grid_search.best_estimator_, X_preprocessed, y)
display = LearningCurveDisplay(train_sizes=train_sizes, train_scores=train_scores, test_scores=test_scores, score_name="Score")
display.plot()

In [ ]:
display.plot(score_type='both')

In [ ]:
df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Challenge_solved_time'].notna()]
df = df[df['Platform'].str.contains('Git')]

param_grid = {
    'n_estimators': [20, 40, 60, 80, 100],
    'max_depth': [3, 6, 9],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'objective': ['reg:squarederror'],
    'random_state': [random_state],
    'tree_method': ['gpu_hist'],
    'gpu_id': [1],
}

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=np.nan
)
numerical_scaler = RobustScaler()
preprocessor = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_scaler, numerical_columns_git),
    ],
    verbose_feature_names_out=False,
)
    
X = df[categorical_columns + numerical_columns_git]
X_preprocessed = preprocessor.fit_transform(X)
y = df['Challenge_solved_time']

regressor = XGBRegressor()
random_search = RandomizedSearchCV(regressor, param_grid, n_jobs=-1, random_state=random_state)
random_search.fit(X_preprocessed,y)
    
name = 'Overall'
print(f'{name} Squared Error: {random_search.best_score_}')
    
explainer = shap.TreeExplainer(random_search.best_estimator_)
shap_values = explainer(X_preprocessed).values.mean(0)
shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
feature_importance = pd.Series(
    shap_values,
    index = X.columns,
).to_frame(name)

roc_auc = []

for name, group in df.groupby('Tool'):
    if len(group) < 50:
        continue

    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=np.nan
    )
    numerical_scaler = RobustScaler()
    preprocessor = ColumnTransformer(
        [
            ("cat", categorical_encoder, categorical_columns),
            ("num", numerical_scaler, numerical_columns_git),
        ],
        verbose_feature_names_out=False,
    )
    
    X = group[categorical_columns + numerical_columns_git]
    X_preprocessed = preprocessor.fit_transform(X)
    y = group['Challenge_solved_time']

    regressor = XGBRegressor()
    random_search = RandomizedSearchCV(regressor, param_grid, n_jobs=-1, random_state=random_state)
    random_search.fit(X_preprocessed,y)
    
    roc_auc.append(random_search.best_score_)
    print(f'{name}: {random_search.best_score_}')
    
    explainer = shap.TreeExplainer(random_search.best_estimator_)
    shap_values = explainer(X_preprocessed).values.mean(0)
    shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
    importances = pd.Series(
        shap_values,
        index = X.columns,
    ).to_frame(name)
    feature_importance = pd.concat([feature_importance, importances], axis=1)

print(f"Average Squared Error: {sum(roc_auc)/len(roc_auc)}")
fig = px.imshow(feature_importance, title='Challenge closed time feature importance (Git)', color_continuous_scale='RdBu', width=1000, height=1000)
fig.write_image(os.path.join(path_rq3, 'Challenge closed time feature importance (Git).png'))


In [ ]:
df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Challenge_solved_time'].notna()]
df = df[df['Platform'].str.contains('Stack Overflow')]

param_grid = {
    'n_estimators': [100, 500, 1000],
    'max_depth': [6, 9, 12, 15],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'objective': ['reg:squarederror'],
    'random_state': [random_state],
    'tree_method': ['gpu_hist'],
    'gpu_id': [1],
}

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=np.nan
)
numerical_scaler = RobustScaler()
preprocessor = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_scaler, numerical_columns_so),
    ],
    verbose_feature_names_out=False,
)
    
X = df[categorical_columns + numerical_columns_so]
X_preprocessed = preprocessor.fit_transform(X)
y = df['Challenge_solved_time']

regressor = XGBRegressor()
random_search = RandomizedSearchCV(regressor, param_grid, n_jobs=-1, random_state=random_state)
random_search.fit(X_preprocessed,y)
    
name = 'Overall'
print(f'{name} Squared Error: {random_search.best_score_}')
    
explainer = shap.TreeExplainer(random_search.best_estimator_)
shap_values = explainer(X_preprocessed).values.mean(0)
shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
feature_importance = pd.Series(
    shap_values,
    index = X.columns,
).to_frame(name)

roc_auc = []

for name, group in df.groupby('Tool'):
    if len(group) < 50:
        continue

    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=np.nan
    )
    numerical_scaler = RobustScaler()
    preprocessor = ColumnTransformer(
        [
            ("cat", categorical_encoder, categorical_columns),
            ("num", numerical_scaler, numerical_columns_so),
        ],
        verbose_feature_names_out=False,
    )
    
    X = group[categorical_columns + numerical_columns_so]
    X_preprocessed = preprocessor.fit_transform(X)
    y = group['Challenge_solved_time']

    regressor = XGBRegressor()
    random_search = RandomizedSearchCV(regressor, param_grid, n_jobs=-1, random_state=random_state)
    random_search.fit(X_preprocessed,y)
    
    roc_auc.append(random_search.best_score_)
    print(f'{name}: {random_search.best_score_}')
    
    explainer = shap.TreeExplainer(random_search.best_estimator_)
    shap_values = explainer(X_preprocessed).values.mean(0)
    shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
    importances = pd.Series(
        shap_values,
        index = X.columns,
    ).to_frame(name)
    feature_importance = pd.concat([feature_importance, importances], axis=1)

print(f"Average Squared Error: {sum(roc_auc)/len(roc_auc)}")
fig = px.imshow(feature_importance, title='Challenge closed time feature importance (Stack Overflow)', color_continuous_scale='RdBu', width=1000, height=1000)
fig.write_image(os.path.join(path_rq3, 'Challenge closed time feature importance (Stack Overflow).png'))


In [ ]:
df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Challenge_solved_time'].notna()]
df = df[df['Platform'].str.contains('Tool-specific')]
df = df[~df['Tool'].isin(tool_wo_accepted_answer)]

param_grid = {
    'n_estimators': [100, 500, 1000],
    'max_depth': [6, 9, 12, 15],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'objective': ['reg:squarederror'],
    'random_state': [random_state],
    'tree_method': ['gpu_hist'],
    'gpu_id': [1],
}

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=np.nan
)
numerical_scaler = RobustScaler()
preprocessor = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_scaler, numerical_columns_to),
    ],
    verbose_feature_names_out=False,
)
    
X = df[categorical_columns + numerical_columns_to]
X_preprocessed = preprocessor.fit_transform(X)
y = df['Challenge_solved_time']

regressor = XGBRegressor()
random_search = RandomizedSearchCV(regressor, param_grid, n_jobs=-1, random_state=random_state)
random_search.fit(X_preprocessed,y)
    
name = 'Overall'
print(f'{name} Squared Error: {random_search.best_score_}')
    
explainer = shap.TreeExplainer(random_search.best_estimator_)
shap_values = explainer(X_preprocessed).values.mean(0)
shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
feature_importance = pd.Series(
    shap_values,
    index = X.columns,
).to_frame(name)

roc_auc = []

for name, group in df.groupby('Tool'):
    if len(group) < 50:
        continue

    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=np.nan
    )
    numerical_scaler = RobustScaler()
    numerical_columns = numerical_columns_to_basic
    if name in numerical_columns_to_customized:
        numerical_columns = numerical_columns + numerical_columns_to_customized[name]
    preprocessor = ColumnTransformer(
        [
            ("cat", categorical_encoder, categorical_columns),
            ("num", numerical_scaler, numerical_columns),
        ],
        verbose_feature_names_out=False,
    )
    
    X = group[categorical_columns + numerical_columns]
    X_preprocessed = preprocessor.fit_transform(X)
    y = group['Challenge_solved_time']

    regressor = XGBRegressor()
    random_search = RandomizedSearchCV(regressor, param_grid, n_jobs=-1, random_state=random_state)
    random_search.fit(X_preprocessed,y)
    
    roc_auc.append(random_search.best_score_)
    print(f'{name}: {random_search.best_score_}')
    
    explainer = shap.TreeExplainer(random_search.best_estimator_)
    shap_values = explainer(X_preprocessed).values.mean(0)
    shap_values = 2 * (shap_values - np.min(shap_values)) / (np.max(shap_values) - np.min(shap_values)) - 1
    importances = pd.Series(
        shap_values,
        index = X.columns,
    ).to_frame(name)
    feature_importance = pd.concat([feature_importance, importances], axis=1)

print(f"Average Squared Error: {sum(roc_auc)/len(roc_auc)}")
feature_importance = feature_importance.fillna(0)
fig = px.imshow(feature_importance, title='Challenge closed time feature importance (Tool-specific)', color_continuous_scale='RdBu', width=1000, height=1000)
fig.write_image(os.path.join(path_rq3, 'Challenge closed time feature importance (Tool-specific).png'))


In [ ]:
df = pd.read_json(os.path.join(path_rq3, 'metrics.json'))
df = df[df['Challenge_solved_time'].notna()]

from plotly.colors import n_colors
# draw the violin plots for the challenge solved time
colors = n_colors('rgb(5, 200, 200)', 'rgb(200, 10, 10)', df['Challenge_summary_topic_macro'].nunique(), colortype='rgb')

# log scale all numerical values for better visualization of long-tailed distributions
df['Challenge_solved_time'] = np.log(df['Challenge_solved_time'] + 1)

fig_challenge_solved_time = go.Figure()

for (name, group), color in zip(df.groupby('Challenge_summary_topic_macro'), colors):
    fig_challenge_solved_time.add_trace(go.Violin(
        x=group['Challenge_solved_time'], y=group['Challenge_summary_topic_macro'], line_color=color))

fig_challenge_solved_time.update_traces(
    orientation='h', meanline_visible=True, side='positive', width=3, points=False)
fig_challenge_solved_time.update_layout(
    height=1000,
    width=2000,
    font=dict(size=20),
    margin=dict(l=0, r=0, t=0, b=0),
    xaxis_showgrid=False,
    xaxis_zeroline=False,
    showlegend=False,
    xaxis_title='Anomaly solved time (hours) (log scale)',
)
fig_challenge_solved_time.write_image(os.path.join(
    path_rq3, 'Anomaly_solved_time.png'))